# Trazabilidad OpenSees - Unity - Capacidad
## P1L4: demostracion reproducible paso a paso

Este notebook demuestra como se relacionan:

```text
elementTag de OpenSees -> ID exportado -> objeto Unity -> resultados -> seccion/capacidad
```

La demostracion utiliza los archivos reales del proyecto. No vuelve a resolver
el edificio: lee los resultados generados por Python/OpenSees y comprueba sus
identificadores, conectividad y archivos asociados.

## 1. Idea general

OpenSees identifica cada elemento con un `elementTag`. Python conserva ese
identificador cuando escribe los resultados. El exportador de Unity vuelve a
usar el mismo ID y Unity crea un objeto cuyo nombre y registro del inspector
lo contienen.

Para una barra, la cadena es directa:

```text
OpenSees elementTag 15
    -> elemento 15 en los CSV/JSON
    -> COLUMN_ID_15 en Unity
    -> fuerzas R:15
    -> seccion y curva P-M del miembro 15
```

Para un muro, la malla ShellMITC4 puede tener varios elementos por paño. La
cadena incluye el `wall_id` del paño por piso:

```text
ShellMITC4 elementTag(s)
    -> wall_id 1002, source_wall_id 10, floor 2
    -> Muro_ID_1002_Piso_2 en Unity
    -> demanda y curva P-M del paño 1002
```

In [1]:
from pathlib import Path
import csv
import json
import re
from pprint import pprint

def locate_edificio():
    """Encuentra Edificio tanto si el notebook parte en la raiz como en notebooks/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'Edificio' / 'results').is_dir():
            return candidate / 'Edificio'
        if candidate.name.lower() == 'edificio' and (candidate / 'results').is_dir():
            return candidate
    raise FileNotFoundError('No se encontro Edificio/results')

EDIFICIO = locate_edificio()
RESULTS = EDIFICIO / 'results'
UNITY = EDIFICIO / 'visualization' / 'unity' / 'UnityVisualization'
RESOURCES = UNITY / 'Assets' / 'Resources'
print('Edificio:', EDIFICIO)
print('Resultados:', RESULTS)
print('Resources Unity:', RESOURCES)

Edificio: c:\Users\benja\OneDrive\Documentos\Universidad\4to Año 2do Semestre\Metodos Computacionales en IOC\Proyecto1\Edificio
Resultados: c:\Users\benja\OneDrive\Documentos\Universidad\4to Año 2do Semestre\Metodos Computacionales en IOC\Proyecto1\Edificio\results
Resources Unity: c:\Users\benja\OneDrive\Documentos\Universidad\4to Año 2do Semestre\Metodos Computacionales en IOC\Proyecto1\Edificio\visualization\unity\UnityVisualization\Assets\Resources


## 2. Archivos que participan

| Etapa | Archivo | Funcion |
| --- | --- | --- |
| Modelo | `results/modelo_3d_manual.json` | Nodos, barras, muros por piso y propiedades geometricas. |
| Exportacion | `Assets/Resources/model_3d.csv` | Datos que `BuildingVisualizer` lee para crear la escena. |
| Respuesta | `results/R_fuerzas_locales.json` | Fuerzas locales del caso R por elemento. |
| Respuesta Unity | `Assets/Resources/semana3_esfuerzos_locales.csv` | Fuerzas locales consultadas por el inspector. |
| Capacidad columna | `Assets/Resources/semana3_graficos_seccion.json` | Secciones, materiales y curva P-M de miembros. |
| Capacidad muro | `results/PM_muros_unity.json` | Curvas P-M, armadura y segmentos por paño. |
| Demanda muro | `results/demanda_muros.csv` | Demanda Shell reducida al corte del paño. |
| Verificacion | `results/verificacion_demanda_capacidad.csv` | Demanda, capacidad y estado elemento por elemento. |
| Codigo Unity | `Assets/Scripts/BuildingVisualizer.cs` y `ElementInspector.cs` | Creacion, registro y consulta del objeto. |

In [2]:
def load_json(path):
    with path.open(encoding='utf-8') as stream:
        return json.load(stream)

def load_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))

model = load_json(RESULTS / 'modelo_3d_manual.json')
pm_walls = load_json(RESULTS / 'PM_muros_unity.json')
capacity = load_json(RESOURCES / 'semana3_graficos_seccion.json')
verification = load_json(RESULTS / 'verificacion_demanda_capacidad.json')
model_csv = load_csv(RESOURCES / 'model_3d.csv')
forces_csv = load_csv(RESOURCES / 'semana3_esfuerzos_locales.csv')
wall_demand = load_csv(RESULTS / 'demanda_muros.csv')

print('Nodos en modelo JSON:', len(model['nodes']))
print('Barras en modelo JSON:', len(model['elements']))
print('Panos de muro:', len(model['walls']))
print('Muros con curvas P-M:', len(pm_walls['walls']))
print('Estado demanda-capacidad:', verification)
print('Filas del CSV consumido por Unity:', len(model_csv))

Nodos en modelo JSON: 1251
Barras en modelo JSON: 694
Panos de muro: 82
Muros con curvas P-M: 82
Estado demanda-capacidad: {'estado': 'OK', 'total': 1650, 'cumplen': 1650, 'fuera_capacidad': 0, 'muros': 410, 'columnas': 1240}
Filas del CSV consumido por Unity: 2708


## 3. Ejemplo A: columna con correspondencia directa

Se utiliza la columna `15`. En este caso el `elementTag`, el ID exportado, el
nombre del objeto Unity y la clave de resultados deben coincidir.

In [3]:
COLUMN_ID = 15

model_column = next(row for row in model['elements'] if int(row['id']) == COLUMN_ID)
unity_column = next(row for row in model_csv if row['kind'] == 'E' and int(row['id']) == COLUMN_ID)
r_column = next(row for row in forces_csv if row['caso'] == 'R' and int(row['elemento']) == COLUMN_ID)
section_member = next(row for row in capacity['members'] if int(row['id']) == COLUMN_ID)

print('1) OpenSees/modelo_3d_manual.json')
pprint(model_column)
print('\n2) Registro E que Unity lee desde model_3d.csv')
pprint(unity_column)
print('\n3) Nombre esperado del objeto Unity')
unity_name = f"{model_column['type']}_ID_{COLUMN_ID}"
print(unity_name)
print('\n4) Resultado del caso R para el mismo ID')
pprint(r_column)
print('\n5) Seccion/capacidad registrada para el mismo ID')
pprint(section_member)

1) OpenSees/modelo_3d_manual.json
{'i': 700006, 'id': 15, 'j': 701006, 'status': 'MANUAL', 'type': 'COLUMN'}

2) Registro E que Unity lee desde model_3d.csv
{'aux': '',
 'axis': None,
 'i': '700006',
 'id': '15',
 'j': '701006',
 'kind': 'E',
 'level': '',
 'restraint': None,
 'status': 'MANUAL',
 'type': 'COLUMN',
 'x_m': '',
 'y_m': '',
 'z_m': ''}

3) Nombre esperado del objeto Unity
COLUMN_ID_15

4) Resultado del caso R para el mismo ID
{'Myi_kNm': '-0.6993262091376924',
 'Myj_kNm': '-1.3986251768422475',
 'Mzi_kNm': '19.58679522707095',
 'Mzj_kNm': '39.1736046796661',
 'Ni_kN': '7346.883309334297',
 'Nj_kN': '-7346.883309334297',
 'Ti_kNm': '-4.4976636600649937e-07',
 'Tj_kNm': '4.4976636600649937e-07',
 'Vyi_kN': '14.8384848249336',
 'Vyj_kN': '-14.8384848249336',
 'Vzi_kN': '0.5297857035302879',
 'Vzj_kN': '-0.5297857035302879',
 'caso': 'R',
 'elemento': '15'}

5) Seccion/capacidad registrada para el mismo ID
{'has_capacity': True, 'id': 15, 'type': 'COLUMN'}


### Interpretacion de la columna

El resultado de la celda anterior demuestra cinco enlaces:

1. El modelo estructural contiene el elemento con ID `15`.
2. Ese ID aparece como registro `E` en el CSV de Unity.
3. `BuildingVisualizer` crea un objeto con nombre `COLUMN_ID_15`.
4. `ElementInspector` registra el objeto con ID `15` y consulta la fila `R,15`.
5. `SectionGraphs` busca el miembro `15` en el JSON de secciones y dibuja su P-M nominal.

El inspector muestra fuerzas de demanda; la curva P-M muestra la capacidad.
No se debe confundir la amplificacion visual de un diagrama con un cambio de
los valores numericos.

## 4. Ejemplo B: muro separado por piso

Se utiliza el paño `1002`. Su identificacion significa:

- `source_wall_id = 10`: muro de origen.
- `floor = 2`: segundo piso del esquema de paños.
- `wall_id = 1002`: ID visual y de capacidad del paño.

Para muros no se debe afirmar que un unico objeto Unity equivale a un unico
ShellMITC4. El exportador relaciona los shells con el paño mediante `wall`;
despues Unity selecciona el paño por su ID.

In [ ]:
WALL_ID = 1002

model_wall = next(row for row in model['walls'] if int(row['id']) == WALL_ID)
unity_wall = next(row for row in pm_walls['walls'] if int(row['id']) == WALL_ID)
wall_rows = [row for row in wall_demand if int(row['panel_id']) == WALL_ID]

print('1) Paño en modelo estructural')
pprint(model_wall)
print('\n2) Datos P-M preparados para Unity')
print({key: unity_wall[key] for key in ('id', 'source_id', 'floor', 'name', 'confidence')})
print('Segmentos de capacidad:', len(unity_wall['segments']))
print('\n3) Demandas exportadas para el paño')
for row in wall_rows[:3]:
    pprint(row)
print('Filas de demanda encontradas:', len(wall_rows))
print('\n4) Nombre esperado del objeto Unity')
print(f"Muro_ID_{WALL_ID}_Piso_{model_wall['floor']}")

### Como se relacionan los ShellMITC4

Durante la construccion del modelo OpenSees, `modelo_opensees_3d.py` crea
elementos `ShellMITC4` con tags propios. El exportador
`exportar_resultados_unity.py` obtiene esos tags con `ops.getEleTags()`,
recupera la conectividad con `ops.eleNodes(tag)` y busca el unico paño cuyas
conectividades contienen esos nodos. El resultado conceptual es:

```text
shell tag  ->  wall_id 1002  ->  source_wall_id 10 / floor 2
```

La evidencia de la demanda conserva cuantos shells participan en el corte en
la columna `shells_corte` de `demanda_muros.csv`. Por eso la demanda de muro
se consulta por paño, mientras que la malla resistente se calcula con varios
ShellMITC4.

In [4]:
export_script = (EDIFICIO / 'visualization' / 'exports' / 'exportar_resultados_unity.py').read_text(encoding='utf-8')
model_script = (EDIFICIO / 'model' / 'opensees' / 'modelo_opensees_3d.py').read_text(encoding='utf-8')

print('Fragmentos que prueban la relacion shell -> muro:')
for line in export_script.splitlines():
    if 'getEleTags' in line or 'eleNodes' in line or 'owners' in line or 'shells.append' in line:
        print('  ', line.strip())
print('\nFragmentos que crean ShellMITC4:')
for line in model_script.splitlines():
    if 'ShellMITC4' in line or 'ops.element' in line and 'next_shell' in line:
        print('  ', line.strip())

Fragmentos que prueban la relacion shell -> muro:
   for tag in sorted(ops.getEleTags()):
   connectivity = list(ops.eleNodes(tag))
   owners = [wall for wall, ids in wall_nodes.items() if set(connectivity) <= ids]
   if len(connectivity) != 4 or len(owners) != 1:
   shells.append(dict(id=tag, wall=owners[0], nodes=connectivity))

Fragmentos que crean ShellMITC4:
   """Modelo OpenSeesPy 3D: vigas, columnas, muros (ShellMITC4) y losas tributarias."""
   # analytical representation is generated below with ShellMITC4.
   # Wall ShellMITC4 mesh
   """Discretise every wall as ShellMITC4 elements.
   ops.element("ShellMITC4", next_shell, n1, n2, n3, n4, section_tag)
   print(f"Muros malla: {wm.get('wall_count', 0)} muros, {wm.get('shell_count', 0)} ShellMITC4, {wm.get('mesh_node_count', 0)} nodos de malla")


## 5. Comparacion demanda-capacidad

La capacidad no se infiere por el color del objeto. Se compara numericamente
en los archivos de resultados:

- Columnas: `resumen_capacidad.json`, `PM_puntos.csv` y
  `semana3_graficos_seccion.json`.
- Muros: `PM_muros_unity.json`, `PM_muros_resumen.csv` y
  `demanda_muros.csv`.
- Resultado consolidado: `verificacion_demanda_capacidad.csv` y `.json`.

El siguiente bloque comprueba los conteos y revisa ejemplos de la tabla final.

In [ ]:
capacity_rows = load_csv(RESULTS / 'verificacion_demanda_capacidad.csv')
print('Resumen:', verification)
print('Columnas en tabla detallada:', len(capacity_rows))
print('Encabezados:', list(capacity_rows[0]))
print('\nPrimeras filas:')
for row in capacity_rows[:3]:
    pprint(row)

assert verification['estado'] == 'OK'
assert verification['cumplen'] == verification['total']
assert verification['fuera_capacidad'] == 0
assert any(int(row.get('elemento', -1)) == COLUMN_ID for row in capacity_rows)
assert any(int(row.get('elemento', -1)) == WALL_ID for row in capacity_rows) or wall_rows
print('\nControles de consistencia: OK')

## 6. Verificacion del codigo de Unity

Ademas de los datos, se puede mostrar en el codigo como se construye el
enlace. Las lineas siguientes extraen los fragmentos relevantes para que la
demostracion no dependa de una captura manual del editor.

In [ ]:
building_cs = (UNITY / 'Assets' / 'Scripts' / 'BuildingVisualizer.cs').read_text(encoding='utf-8')
inspector_cs = (UNITY / 'Assets' / 'Scripts' / 'ElementInspector.cs').read_text(encoding='utf-8')
wall_graph_cs = (UNITY / 'Assets' / 'Scripts' / 'WallSectionGraphs.cs').read_text(encoding='utf-8')

patterns = {
    'BuildingVisualizer': r'CreateMember|CreateWall|inspector\.Register|Muro_ID_',
    'ElementInspector': r'public void SelectElementById|TryForces|semana3_esfuerzos_locales',
    'WallSectionGraphs': r'semana3_pm_muros|semana4_resultados|wall\.id|wallDemand',
}
sources = {'BuildingVisualizer': building_cs, 'ElementInspector': inspector_cs, 'WallSectionGraphs': wall_graph_cs}
for name, pattern in patterns.items():
    print('\n' + name)
    for number, line in enumerate(sources[name].splitlines(), 1):
        if re.search(pattern, line):
            print(f'{number:4}: {line.strip()}')

## 7. Guion para demostrarlo en Unity

1. Abrir `Edificio/visualization/unity/UnityVisualization` en Unity.
2. Cargar `Assets/Main.unity` y comprobar que el modelo no esta vacio.
3. Buscar o seleccionar `Columna 15`.
4. Mostrar en el inspector el ID `15` y las fuerzas del caso `R`.
5. Contrastar esas fuerzas con la fila `R,15` de
   `Assets/Resources/semana3_esfuerzos_locales.csv`.
6. Abrir la pestaña de capacidad y mostrar la curva P-M de la seccion.
7. Repetir con `Muro 1002`, verificando `source_wall_id = 10` y `floor = 2`.
8. Mostrar la curva P-M del paño y la demanda seleccionada.
9. Contrastar el estado final con
   `results/verificacion_demanda_capacidad.json`.

La captura ideal debe mostrar simultaneamente el ID del inspector, el panel de
resultados y la curva de capacidad. Las tablas CSV/JSON son la evidencia
numerica que permite comprobar lo que se ve en pantalla.

## 8. Limitaciones que deben declararse

- Unity funciona como postprocesador; no vuelve a resolver OpenSees.
- La deformada y los diagramas pueden utilizar una escala visual.
- En muros, un paño Unity puede representar varios ShellMITC4.
- La curva P-M de columna corresponde a la seccion exportada y su asignacion;
  no reemplaza un diseño completo.
- La curva P-M de muro es nominal y depende de la armadura adoptada.
- La prueba del notebook verifica archivos y relaciones de IDs; la evidencia
  visual final requiere ejecutar la escena Unity.

## Conclusion

La trazabilidad queda demostrada cuando el ID se puede seguir sin cambiar de
significado desde el modelo estructural hasta el objeto Unity, sus resultados
y su seccion/capacidad. Para barras la correspondencia es directa; para muros
se documenta explicitamente la agrupacion de elementos ShellMITC4 en paños por
piso.